# megane - Pipeline API Demo

Build visualization workflows programmatically using the Pipeline API.
Each pipeline is a directed graph of nodes connected by edges, ending at an explicit Viewport node.

In [ ]:
import megane

print(f"megane v{megane.__version__}")

## 1. Basic Structure

Load a PDB file and display it. The simplest pipeline connects `LoadStructure` to a `Viewport`.

In [ ]:
pipe = megane.Pipeline()
s = pipe.add_node(megane.LoadStructure("../tests/fixtures/1crn.pdb"))
v = pipe.add_node(megane.Viewport())

pipe.add_edge(s.out.particle, v.inp.particle)

viewer = megane.MolecularViewer()
viewer.set_pipeline(pipe)
viewer

## 2. Structure with Bonds

Add distance-based bond detection with `AddBonds`.

In [ ]:
pipe = megane.Pipeline()
s = pipe.add_node(megane.LoadStructure("../tests/fixtures/caffeine_water.pdb"))
bonds = pipe.add_node(megane.AddBonds(source="distance"))
v = pipe.add_node(megane.Viewport())

pipe.add_edge(s.out.particle, bonds.inp.particle)
pipe.add_edge(s.out.particle, v.inp.particle)
pipe.add_edge(bonds.out.bond, v.inp.bond)

viewer = megane.MolecularViewer()
viewer.set_pipeline(pipe)
viewer

## 3. Filter and Modify

Select carbon atoms with `Filter`, then scale them up with `Modify`.

In [ ]:
pipe = megane.Pipeline()
s = pipe.add_node(megane.LoadStructure("../tests/fixtures/1crn.pdb"))
carbons = pipe.add_node(megane.Filter(query="element == 'C'"))
big = pipe.add_node(megane.Modify(scale=1.5, opacity=0.8))
bonds = pipe.add_node(megane.AddBonds(source="distance"))
v = pipe.add_node(megane.Viewport())

pipe.add_edge(s.out.particle, carbons.inp.particle)
pipe.add_edge(carbons.out.particle, big.inp.particle)
pipe.add_edge(s.out.particle, bonds.inp.particle)
pipe.add_edge(big.out.particle, v.inp.particle)
pipe.add_edge(bonds.out.bond, v.inp.bond)

viewer = megane.MolecularViewer()
viewer.set_pipeline(pipe)
viewer

## 4. Trajectory Playback

Load a structure with an XTC trajectory. Use `frame_index` to navigate frames.

In [ ]:
pipe = megane.Pipeline()
s = pipe.add_node(megane.LoadStructure("../tests/fixtures/caffeine_water.pdb"))
t = pipe.add_node(megane.LoadTrajectory(xtc="../tests/fixtures/caffeine_water_vibration.xtc"))
bonds = pipe.add_node(megane.AddBonds(source="structure"))
v = pipe.add_node(megane.Viewport())

pipe.add_edge(s.out.particle, t.inp.particle)
pipe.add_edge(s.out.particle, bonds.inp.particle)
pipe.add_edge(s.out.particle, v.inp.particle)
pipe.add_edge(t.out.traj, v.inp.traj)
pipe.add_edge(bonds.out.bond, v.inp.bond)

viewer = megane.MolecularViewer()
viewer.set_pipeline(pipe)
print(f"Loaded trajectory: {viewer.total_frames} frames")
viewer

In [ ]:
# Jump to a specific frame
viewer.frame_index = 50

## 5. Multiple Structures

Load two structures into the same viewport as independent layers.

In [ ]:
pipe = megane.Pipeline()
s1 = pipe.add_node(megane.LoadStructure("../tests/fixtures/1crn.pdb"))
s2 = pipe.add_node(megane.LoadStructure("../tests/fixtures/caffeine_water.pdb"))
b1 = pipe.add_node(megane.AddBonds(source="distance"))
b2 = pipe.add_node(megane.AddBonds(source="distance"))
v = pipe.add_node(megane.Viewport())

pipe.add_edge(s1.out.particle, b1.inp.particle)
pipe.add_edge(s2.out.particle, b2.inp.particle)
pipe.add_edge(s1.out.particle, v.inp.particle)
pipe.add_edge(b1.out.bond, v.inp.bond)
pipe.add_edge(s2.out.particle, v.inp.particle)
pipe.add_edge(b2.out.bond, v.inp.bond)

viewer = megane.MolecularViewer()
viewer.set_pipeline(pipe)
viewer

## 6. Labels

Add element labels to atoms with `AddLabels`.

In [ ]:
pipe = megane.Pipeline()
s = pipe.add_node(megane.LoadStructure("../tests/fixtures/caffeine_water.pdb"))
bonds = pipe.add_node(megane.AddBonds(source="distance"))
labels = pipe.add_node(megane.AddLabels(source="element"))
v = pipe.add_node(megane.Viewport())

pipe.add_edge(s.out.particle, bonds.inp.particle)
pipe.add_edge(s.out.particle, labels.inp.particle)
pipe.add_edge(s.out.particle, v.inp.particle)
pipe.add_edge(bonds.out.bond, v.inp.bond)
pipe.add_edge(labels.out.label, v.inp.label)

viewer = megane.MolecularViewer()
viewer.set_pipeline(pipe)
viewer

## 7. Import / Export

Pipelines can be saved to and loaded from JSON files for reproducible workflows.

### Browser
In the Pipeline Editor panel, click **Export** to download the current pipeline as `pipeline.json`,
or **Import** to upload a saved JSON file and restore it in the editor.

### Python API
- `pipeline.to_json()` — serialize to a JSON string
- `pipeline.save(path)` — write JSON to a file
- `Pipeline.from_dict(d)` — reconstruct from a plain dict
- `Pipeline.from_json(s)` — reconstruct from a JSON string
- `Pipeline.load(path)` — load from a JSON file

In [ ]:
import tempfile, pathlib

# Build a pipeline to export
pipe = megane.Pipeline()
s = pipe.add_node(megane.LoadStructure("../tests/fixtures/1crn.pdb"))
bonds = pipe.add_node(megane.AddBonds(source="structure"))
v = pipe.add_node(megane.Viewport())

pipe.add_edge(s.out.particle, bonds.inp.particle)
pipe.add_edge(s.out.particle, v.inp.particle)
pipe.add_edge(bonds.out.bond, v.inp.bond)

# Serialize to JSON string
json_str = pipe.to_json()
print(json_str[:300], "...")

In [ ]:
# Save to a file and load it back
tmp = pathlib.Path(tempfile.mkdtemp()) / "my_pipeline.json"
pipe.save(tmp)
print(f"Saved to {tmp}")

# Load from file — file paths must still be accessible
pipe2 = megane.Pipeline.load(tmp)

viewer2 = megane.MolecularViewer()
viewer2.set_pipeline(pipe2)
viewer2

In [ ]:
# Round-trip via dict (from_dict / from_json also work)
pipe3 = megane.Pipeline.from_dict(pipe.to_dict())

viewer3 = megane.MolecularViewer()
viewer3.set_pipeline(pipe3)
viewer3